# 3- Separación de variables X,Y

In [25]:
y = df.Churn
X = df[['Age','Gender','Tenure','ContractType','InternetService','MonthlyCharges']]

In [26]:
y.head()

,Churn
0,Yes
1,Yes
2,Yes
3,Yes
4,Yes


In [27]:
X.head()

,Age,Gender,Tenure,ContractType,InternetService,MonthlyCharges
0,49,Male,4,Month-to-Month,Fiber Optic,88.35
1,43,Male,0,Month-to-Month,Fiber Optic,36.67
2,51,Female,2,Month-to-Month,Fiber Optic,63.79
3,60,Female,8,One-Year,DSL,102.34
4,42,Male,32,Month-to-Month,,69.01


# 4- Train Split (X,y)

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((800, 6), (200, 6), (800,), (200,))

# 5- Encoding

In [29]:
# Genero | X
# Mujer = 1 | Hombre = 0

X_train.Gender = X_train['Gender'].apply(lambda x: 1 if x == 'Female' else 0)
X_test.Gender = X_test['Gender'].apply(lambda x: 1 if x == 'Female' else 0)

In [30]:
# Churn | y
# Yes = 1 | No = 0
y_train = y_train.apply(lambda x: 1 if x == 'Yes' else 0)


In [31]:
y_test = y_test.apply(lambda x: 1 if x == 'Yes' else 0)

In [32]:
# One-Hot Encoding a ContractType y InternetService

ohe = OneHotEncoder(sparse_output=False, drop='first', dtype=int)

encoded_train = ohe.fit_transform(X_train[['ContractType', 'InternetService']])
encoded_test = ohe.transform(X_test[['ContractType','InternetService']])

df_enc_train = pd.DataFrame(encoded_train, columns=ohe.get_feature_names_out(), index=X_train.index)
df_enc_test = pd.DataFrame(encoded_test, columns=ohe.get_feature_names_out(), index=X_test.index)

In [33]:
X_train = pd.concat([X_train, df_enc_train], axis='columns')
X_train = X_train.drop(columns=['ContractType','InternetService'], axis='columns')

X_test = pd.concat([X_test, df_enc_test], axis='columns')
X_test = X_test.drop(columns=['ContractType','InternetService'], axis='columns')

In [34]:
X_train.head()

,Age,Gender,Tenure,MonthlyCharges,ContractType_One-Year,ContractType_Two-Year,InternetService_DSL,InternetService_Fiber Optic
29,42,0,62,97.66,0,1,0,1
535,45,1,21,102.76,1,0,0,1
695,41,0,32,49.00,1,0,0,1
557,40,1,50,114.46,0,0,0,0
836,60,1,53,89.57,0,0,0,1


# 6- Escalado de datos

Procedemos a escalar los datos, ya que tenemos campos, los cuales tienen rangos de diferentes magnitudes a comparación de otros (una vez el OHE aplicado, lo cual se hizo en la anterior sección), como por ejemplo: Genero, ContracType (OHE), etc. Esto hará que estén en un rango similar entre todos, es decir, evita el sesgo; hace que un '1' (de genero por ejemplo) el modelo no interprete que sea menor que un 100 (de MonthlyCharges), si no que tengan un rango similar.

In [35]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [36]:
t_s = pd.DataFrame(X_train_scaled)
t_s.describe()

,0,1,2,3,4,5,6,7
count,8.000000e+02,8.000000e+02,8.000000e+02,8.000000e+02,8.000000e+02,8.000000e+02,8.000000e+02,8.000000e+02
mean,-3.419487e-16,2.442491e-17,2.664535e-17,3.730349e-16,-2.664535e-17,-7.549517e-17,-6.661338e-17,4.107825e-17
std,1.000626e+00,1.000626e+00,1.000626e+00,1.000626e+00,1.000626e+00,1.000626e+00,1.000626e+00,1.000626e+00
min,-3.339463e+00,-1.067257e+00,-9.974118e-01,-1.757359e+00,-6.371620e-01,-5.019524e-01,-6.859943e-01,-7.849596e-01
25%,-6.873753e-01,-1.067257e+00,-7.330392e-01,-8.595712e-01,-6.371620e-01,-5.019524e-01,-6.859943e-01,-7.849596e-01
50%,2.664838e-02,9.369815e-01,-3.364803e-01,2.476039e-02,-6.371620e-01,-5.019524e-01,-6.859943e-01,-7.849596e-01
75%,6.386687e-01,9.369815e-01,3.905445e-01,8.385473e-01,1.569460e+00,-5.019524e-01,1.457738e+00,1.273951e+00
max,3.086750e+00,9.369815e-01,5.453280e+00,1.802922e+00,1.569460e+00,1.992221e+00,1.457738e+00,1.273951e+00


In [37]:
# Almacenamos la instancia:

import joblib
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']